In [1]:
#**1_data_ingestion**

##**Install Dependencies**
"""

!pip uninstall -y datasets
!pip install datasets==2.14.7
!pip install pyspark
!pip install pyarrow

"""##**Spark Session Configuration**"""

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviewsBigData") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

spark

"""##**Load Dataset (Streaming)**"""

from datasets import load_dataset

dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Electronics",
    split="full",
    streaming=True
)

dataset

"""##**Sample ~1.5 Million Records**"""

import itertools
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

batch = list(itertools.islice(dataset, 200000))  # take 200k rows first
pdf = pd.DataFrame(batch)
spark_df = spark.createDataFrame(pdf)

spark_df.show(5)

"""##**Validate Big Data Size**"""

print("Total Rows:", spark_df.count())
spark_df.printSchema()

"""##**Repartition for Parallelism**"""

spark_df = spark_df.repartition(200)
print("Partitions:", spark_df.rdd.getNumPartitions())

spark_df.write.mode("overwrite").parquet("data/amazon_reviews_parquet")

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/content/data/amazon_reviews_parquet. SQLSTATE: 42K03